# Agent Identity SDK: OAuth2 Usage Example

This notebook demonstrates how to interactively use the `agent-identity-dev-sdk` to automatically manage workload access tokens and seamlessly perform an OAuth2 `USER_FEDERATION` flow.

### Overview
The OAuth2 flow requires a server to handle the callback from the identity provider. We will spin up a lightweight FastAPI server in the background directly inside this notebook to handle the redirect and complete the session binding.

### Prerequisites
Before running the remaining cells, ensure you have set your **Huawei Cloud AK/SK** credentials and the desired region.

In [ ]:
import os

# Manually set your credentials here for testing if not already in your environment.
# Ensure you DO NOT commit these secrets to version control.

# os.environ["HUAWEICLOUD_SDK_AK"] = "your-access-key-id"
# os.environ["HUAWEICLOUD_SDK_SK"] = "your-secret-access-key"
# os.environ["HUAWEICLOUD_SDK_REGION"] = "ap-southeast-4"
# os.environ["HUAWEICLOUD_SDK_PROJECT_ID"] = "your-project-id"
# os.environ["HUAWEICLOUD_SDK_DOMAIN_ID"] = "your-domain-id"

print("Environment variables ready.")

In [ ]:
import uuid
import threading
import uvicorn
from typing import Optional, Any, Dict
from fastapi import FastAPI, Request, HTTPException
from agentarts.sdk import (
    AgentArtsRuntimeContext,
    IdentityClient,
    require_access_token,
)
from agentarts.sdk.identity.types import OAuth2Vendor
from huaweicloudsdkagentidentity.v1 import AuthorizerType
from huaweicloudsdkagentidentity.v1.model import UserIdentifier

### Step 1: Set the Context & Initialize Identity Client

We generate unique identifiers for this session and initialize our `IdentityClient`.

In [ ]:
# 0. Generate random identifiers for testing
random_suffix = uuid.uuid4().hex[:8]
user_id = f"user-{random_suffix}"
workload_name = f"oauth-workload-{random_suffix}"
provider_name = f"google-oauth-{random_suffix}"

# 1. Set the user ID for context
AgentArtsRuntimeContext.set_user_id(user_id)

# 2. Initialize client
region = os.getenv("HUAWEICLOUD_SDK_REGION", "ap-southeast-4")
client = IdentityClient(region=region, ignore_ssl_verification=True)
print(f"Context set for User ID: {user_id}")

### Step 2: Create OAuth2 Credential Provider

We need to register our OAuth2 provider. **Important:** Creating the OAuth2 provider will generate a `Callback URL`. You must copy this URL and configure it in your Identity Provider's dashboard (e.g., Google Cloud Console) under "Authorized redirect URIs".

In [ ]:
# 3. Create OAuth2 Credential Provider
print(f"Ensuring '{provider_name}' OAuth2 credential provider exists...")
try:
    provider = client.create_oauth2_credential_provider(
        name=provider_name,
        vendor=OAuth2Vendor.GOOGLEOAUTH2,
        client_id="dummy-client-id",
        client_secret="dummy-client-secret",
    )
    print(f"Created '{provider_name}' OAuth2 credential provider.")

    print("\n=== OAUTH2 CONFIGURATION REQUIRED ===")
    print(f"Callback URL: {provider.callback_url}")
    print(
        "ACTION: Please add this Callback URL to your Identity Provider (e.g., Google Cloud Console) as an Authorized Redirect URI."
    )
    print("=======================================\n")

except Exception as e:
    print(f"OAuth2 credential provider might already exist or creation failed: {e}")

### Step 3: Create Workload Identity

We create a workload identity and configure the allowed callback URLs.

In [ ]:
# 4. Create a workload identity
print(f"Creating workload identity: {workload_name}...")
workload = client.create_workload_identity(
    name=workload_name,
    authorizer_type=AuthorizerType.NONE,
    allowed_resource_oauth2_return_urls=["http://localhost:8000/callback"],
)
print(f"Created Workload: {workload.name}")

### Step 4: Fetch Workload Access Token

We fetch the workload access token and set it in the context.

In [ ]:
# 5. Get and set workload access token
print("Getting workload access token...")
token = client.create_workload_access_token(
    workload_name=workload_name, user_id=user_id
)
AgentArtsRuntimeContext.set_workload_access_token(token)
print("Workload access token successfully retrieved and set in context.")

### Step 5: Setup the Background Callback Server

We need to listen for the redirect from the Identity Provider. We'll set up a FastAPI server to handle the `/callback` endpoint.

In [ ]:
app = FastAPI()
BASE_URL = "http://localhost:8000"
CALLBACK_URL = f"{BASE_URL}/callback"

# Register the callback URL in our context
AgentArtsRuntimeContext.set_oauth2_callback_url(CALLBACK_URL)


@app.get("/callback")
async def oauth_callback(request: Request) -> Dict[str, Any]:
    params = dict(request.query_params)
    session_uri = params.get("session_uri")

    if not session_uri:
        return {"error": "Missing 'session_uri' parameter"}

    try:
        print(f"\n[Server] Binding user {user_id} to session {session_uri}")
        client.complete_resource_token_auth(
            session_uri=session_uri, user_identifier=UserIdentifier(user_id=user_id)
        )
        print("[Server] Binding complete! You can return to the notebook.")
        return {
            "status": "success",
            "message": "User successfully bound. Return to notebook.",
        }
    except Exception as e:
        print(f"[Server] Error binding token: {e}")
        raise HTTPException(status_code=500, detail=str(e))


# Start the server in a background thread
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")


server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
print(f"Background callback server running on {BASE_URL}")

### Step 6: Define the Target Function

We define our function decorated with `@require_access_token`. We also define a handler `handle_auth_url` to display the authorization URL when the SDK needs the user to authenticate.

In [ ]:
def handle_auth_url(auth_url: str) -> None:
    print("\n=== ACTION REQUIRED ===")
    print("Please click the following URL to authorize the agent:")
    print(auth_url)
    print("=======================\n")


@require_access_token(
    provider_name=provider_name,
    scopes=["https://www.googleapis.com/auth/userinfo.email"],
    auth_flow="USER_FEDERATION",
    on_auth_url=handle_auth_url,
    ignore_ssl_verification=True,
)
async def fetch_protected_data(access_token: Optional[str] = None) -> Dict[str, Any]:
    print(
        f"Successfully injected OAuth2 Access Token: {access_token[:10] if access_token else 'None'}..."
    )
    return {"data": "Sensitive data retrieved with OAuth2 token", "token": access_token}

### Step 7: Trigger the Flow

We call the function. Because it's the first time, it will generate an authorization URL and wait for you to visit it. Once you complete the dummy authorization flow, the callback server will handle the redirect, bind the session, and the function below will automatically resume and print the token!

In [ ]:
session_uri = "urn:uuid:dummy-session-id"
AgentArtsRuntimeContext.set_oauth2_custom_state(session_uri)

print("Calling protected function (will block waiting for authorization)...")
result = await fetch_protected_data()
print("\nFunction execution complete!")
print("Result:", result)